# Init

In [ ]:
#Imports
import os
import numpy as np
import yaml
import shutil
import re

def get_directories(directory, regex_search=""):
    """
    This function returns a list of directories from the specified directory that match the regular expression search pattern.
    
    Parameters:
    directory (str): The directory path where to look for directories.
    regex_search (str, optional): The regular expression pattern to match. Default is an empty string, which means all directories are included.
    
    Returns:
    list: A list of directory names that match the regular expression search pattern.
    """
    directories = None
    if os.path.exists(directory):
        directories = [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name)) and len(re.findall(regex_search, name))>0]
    else:
        print(f"Directory does not exist: {directory}")
    return directories

def dir_exist_create(directory):
    # Check if the directory exists
    if not os.path.exists(directory):
        # Create the directory
        os.makedirs(directory)

def create_dirs(dirs):
    new_path = dirs[0]
    for path_part in dirs[1:]:
        new_path = os.path.join(new_path, path_part)
        dir_exist_create(new_path)
    return new_path

def make_list_ifnot(string_or_list):
    return [string_or_list] if type(string_or_list) != list else string_or_list

def load_all(root_dir, wanted_animal_ids=["all"], wanted_session_ids=["all"], print_loading=True):
    """
    Loads animal data from the specified root directory for the given animal IDs.

    Parameters:
    - root_dir (string): The root directory path where the animal data is stored.
    - animal_ids (list, optional): A list of animal IDs to load. Default is ["all"].
    Returns:
    - animals_dict (dict): A dictionary containing animal IDs as keys and corresponding Animal objects as values.
    """
    present_animal_ids = get_directories(root_dir, regex_search="DON-")
    animals_dict = {}

    # Search for animal_ids
    for animal_id in present_animal_ids:
        if animal_id in wanted_animal_ids or "all" in wanted_animal_ids:
            sessions_path = os.path.join(root_dir, animal_id)
            present_sessions = get_directories(sessions_path)
            yaml_file_name = os.path.join(root_dir, animal_id, f"{animal_id}.yaml")
            animal = Animal(yaml_file_name, print_loading=print_loading)
            Animal.root_dir = root_dir
            # Search for 2P Sessions
            for session in present_sessions:
                if session in wanted_session_ids or "all" in wanted_session_ids:
                    animal.get_session_data(session, print_loading=print_loading)
            animals_dict[animal_id] = animal
    return animals_dict

#Classes
class Animal:
    """
    This class represents an animal in an experiment.

    Attributes:
    root_dir (str): The root directory where the data is stored.
    sessions (dict): A dictionary to store session objects for this animal.
    cohort_year (int): The year of the cohort that the animal belongs to.
    dob (str): The date of birth of the animal.
    animal_id (str): The ID of the animal.
    pdays (list of int): The postnatal days when the sessions were conducted.
    session_dates (list of str): The dates when the sessions were conducted.
    session_names (list of str): The names of the sessions.
    sex (str): The sex of the animal.

    Methods:
    load_data(yaml_path): Loads metadata for the animal from a YAML file.
    get_session_data(session_id, print_loading=True): Loads data for a specific session.
    """
    root_dir = "/scicore/projects/donafl00-calcium/Users/Sergej/Steffen_Experiments" 

    def __init__(self, yaml_file_path, print_loading=True) -> None:
        self.sessions = {}
        self.cohort_year = None
        self.dob = None
        self.animal_id = None 
        self.pdays = None 
        self.session_dates = None 
        self.session_names = None 
        self.sex = None 
        self.load_data(yaml_file_path)
        self.animal_dir = os.path.join(Animal.root_dir, self.animal_id)
        if print_loading:
            print(f"Loaded animal: {self.animal_id}")

    def load_data(self, yaml_path):
        with open(yaml_path, "r") as yaml_file:
            animal_metadata_dict = yaml.safe_load(yaml_file)
        self.cohort_year = int(animal_metadata_dict["cohort_year"])
        self.dob = animal_metadata_dict["dob"]
        self.animal_id = animal_metadata_dict["name"]
        self.pdays = [int(pday) for pday in animal_metadata_dict["pdays"]]
        self.session_dates = animal_metadata_dict["session_dates"]
        self.session_names = animal_metadata_dict["session_names"]
        self.sex = animal_metadata_dict["sex"]
    
    def get_session_data(self, session_id, print_loading=True):
        yaml_file_index = self.session_names.index(session_id)
        session = Session(self.animal_id, session_id, 
                          age=self.pdays[yaml_file_index], 
                          session_date=self.session_dates[yaml_file_index], 
                          print_loading=print_loading)
        self.sessions[session_id] = session

class Session:
    cabincorr_fname = "binarized_traces.npz"

    def __init__(self, animal_id, session_id, age, session_date, print_loading=True):
        if print_loading:
            print(f"Loading session: {animal_id} {session_id}")
        self.animal_id = animal_id
        self.session_id = session_id
        self.session_date = session_date
        self.age = age
        self.session_dir = os.path.join(Animal.root_dir, animal_id, session_id)
        self.cabincorr_fpath = os.path.join(self.session_dir, "002P-F", "tif", "suite2p_merged", "plane0", Session.cabincorr_fname)
        self.F_detrended, self.F_upphase = None, None

    def load_cabincorr_data(self):
        bin_traces_zip = None
        if os.path.exists(self.cabincorr_fpath):
            bin_traces_zip = np.load(self.cabincorr_fpath, allow_pickle=True)
        else:
            print(f"No CaBincorrPath found in {self.cabincorr_fpath}")
        return bin_traces_zip
    
    def load_fluoresence(self):
        if type(self.F_detrended) != np.ndarray:
            bin_traces_zip = self.load_cabincorr_data()
            self.F_detrended = bin_traces_zip["F_detrended"] if bin_traces_zip else None
            self.F_upphase = bin_traces_zip["F_upphase"] if bin_traces_zip else None
        return self.F_detrended, self.F_upphase
    
class information_extractor:
    def __init__(self, root_dir, save_dir="extracted", wanted_animal_ids=["all"], wanted_session_ids=["all"], print_loading=True):
        self.root_dir = root_dir
        self.save_path = os.path.join(root_dir, save_dir)
        dir_exist_create(self.save_path)
        self.animals = load_all(root_dir, wanted_animal_ids=wanted_animal_ids,
                            wanted_session_ids=wanted_session_ids, 
                            print_loading=print_loading)
        
    def from_cabincorr(self, data_types):
        data_types = make_list_ifnot(data_types)
        for animal_id, animal in self.animals.items():
            for session_id, session in animal.sessions.items():
                bin_traces_zip = session.load_cabincorr_data()
                if not bin_traces_zip:
                    print(f"No CaBincorrPath found in {session.cabincorr_fpath}")
                    continue
                for data_type in data_types:
                    data = bin_traces_zip[data_type]
                    create_dirs([self.save_path, animal_id, session_id])
                    save_file_path = os.path.join(self.save_path, animal_id, session_id, data_type+".npy")
                    if not os.path.exists(save_file_path):
                        print(f"saving {data_type} to {save_file_path}")
                        np.save(save_file_path, data)
                    else:
                        print(f"File already present {save_file_path}")

    def cabincorr(session, save_dir):#FIXME: not fitted to the class
        if os.path.exists(session.cabincorr_data_path):
            new_path = create_dirs([Animal.root_dir, save_dir, session.animal_id, session.session_id])
            new_file_path = os.path.join(new_path, Session.cabincorr_fname)
            if os.path.exists(new_file_path):
                print(f"File already present {new_file_path}")
            else:
                shutil.copy(session.cabincorr_data_path, new_path)
                print(f"Copied {Session.cabincorr_fname} to {new_path}")
        else:
            print(f"No {Session.cabincorr_fname} found in {session.cabincorr_data_path}")

    def yaml(animal, save_dir): #FIXME: not fitted to the class
        yaml_fname = animal.animal_id+".yaml"
        old_path = os.path.join(animal.animal_dir, yaml_fname)
        new_path = create_dirs([Animal.root_dir, save_dir, animal_id])
        new_file_path = os.path.join(new_path, yaml_fname)
        if os.path.exists(new_file_path):
            print(f"File already present {new_file_path}")
        else:
            shutil.copy(old_path, new_path)
            print(f"Copied {yaml_fname} to {new_path}")

    def create_rodrigo_folder(wanted_animal_ids, root_dir, animals=None): #FIXME: not fitted to the class
        #create folders for rodrigo
        if not animals:
            animals = load_all(root_dir, wanted_animal_ids=wanted_animal_ids, generate=False, print_loading=False) # Load all animals
        corr_fname = "allcell_clean_corr_pval_zscore.npy"
        rodrigo_corr_path = os.path.join(root_dir, "rodrigo")
        dir_exist_create(rodrigo_corr_path)
        for animal_id, animal in animals.items():
            for session_id, session in animal.sessions.items():
                # search for allcell_clean_corr_pval_zscore
                corr_path = None
                for s2p_path in session.suite2p_paths:
                    if "merged" in s2p_path:
                        corr_path = search_file(s2p_path, corr_fname)
                if corr_path:
                    # create directories
                    animal_path = os.path.join(rodrigo_corr_path, animal_id)
                    session_path = os.path.join(animal_path, session_id)
                    dir_exist_create(animal_path)
                    dir_exist_create(session_path)
                    #copy allcell_clean_corr_pval_zscore to created dir
                    shutil.copyfile(corr_path, os.path.join(session_path, corr_fname))

# Usage

In [ ]:
root_dir = "/scicore/projects/donafl00-calcium/Users/Sergej/Steffen_Experiments" 
Animal.root_dir = root_dir #FIXME: integrate into extractor
mice21 = ["DON-002865", "DON-003165", "DON-003343", "DON-006084", "DON-006085", "DON-006087"]
mice22 = ["DON-008497", "DON-008498", "DON-008499", "DON-009191", "DON-009192", "DON-010473", "DON-010477"]
mice23 = ["DON-014837", "DON-014838", "DON-014840", "DON-014847", "DON-014849", "DON-015078", "DON-015079"]
wanted_animal_ids = mice21+mice22
ex = information_extractor(root_dir, wanted_animal_ids=wanted_animal_ids, print_loading=False)
ex.from_cabincorr(["F_detrended", "F_upphase"])